# Phase 2: the first actual relational batches

This checks **untrained** forward-pass equivalence on the pinned RelBench `rel-stack/user-badge`
database. It retains all tables/relationships, uses constant features plus relative-time encoding,
and runs the same weights and sampled occurrences through reference, HGAM, and block execution.
No predictive-quality or end-to-end acceleration claim is being tested yet.

1. Save a copy of this notebook to Drive.
2. Choose **Runtime → Change runtime type → T4 GPU**, leave the other settings at their defaults,
   save, and connect. For a CPU-only diagnostic, change `DEVICE` below to `"cpu"`.
3. In **Secrets**, enable this notebook's access to your existing read-only `GITHUB_TOKEN`.
4. Run the cells in order. This notebook uses **temporary local storage** and does not mount Drive.
5. Run the final ZIP-download cell even if the data check fails, then disconnect the runtime.

The data step downloads the full Parquet files for one public database (substantially larger than
small manifest files) and one validation cohort. Column projection reduces RAM use when reading;
it does not reduce the download size. No separate Hugging Face token is required for this public data.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

from google.colab import userdata

PROJECT_ROOT = Path("/content/temporal-relational-gnn")
PROJECT_REF = "main"  # Replace with a recorded commit for a later reproduction.
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

token = userdata.get("GITHUB_TOKEN")
try:
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            'print("x-access-token" if "username" in sys.argv[1].lower() '
            'else os.environ["TRGNN_CLONE_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        clone_environment = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "TRGNN_CLONE_TOKEN": token,
        }
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        subprocess.run(command, env=clone_environment, check=True)
finally:
    token = None
    if "clone_environment" in globals():
        clone_environment.pop("TRGNN_CLONE_TOKEN", None)
subprocess.run(["git", "checkout", PROJECT_REF], cwd=PROJECT_ROOT, check=True)
if PROJECT_REF == "main":
    subprocess.run(["git", "merge", "--ff-only", "origin/main"], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, check=True)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    check=True,
)

Create a temporary run directory and a live log writer. Outputs will be lost if the VM is deleted
before you download them. Copy the active unit/hour rate and remaining units from Colab's resource
panel if available; otherwise leave them `None` rather than entering an invented value.

In [ ]:
import json
import uuid
from datetime import datetime, timezone

DEVICE = "cuda"
COMPUTE_UNITS_PER_HOUR = None
COMPUTE_UNITS_REMAINING = None
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
RUN_DIR = Path("/content/temporary-trgnn-real/runs") / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "colab-budget.json").write_text(
    json.dumps(
        {
            "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
            "units_remaining_observed": COMPUTE_UNITS_REMAINING,
            "planned_total_units": 200,
            "reserve_units": 50,
        },
        indent=2,
    )
)
print(f"Run directory: {RUN_DIR}")


def run_logged(arguments, log_name):
    with (RUN_DIR / log_name).open("w") as log:
        with subprocess.Popen(
            arguments,
            cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, arguments)

Install the matching binary sampler and verify the runtime **before** downloading real data.
These project commands run in fresh Python processes, so they see the newly installed extension.
If this fails, export the logs and resolve it before continuing.

In [ ]:
run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
arguments = [
    sys.executable,
    "-m",
    "trgnn.preflight",
    "--check-sampler",
    "--output",
    str(RUN_DIR / "preflight.json"),
]
if DEVICE == "cuda":
    arguments.append("--require-cuda")
run_logged(arguments, "preflight.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")

Download pinned inputs, build the projected historical graph, and compare three small real batches.
The first run may spend several minutes downloading and building the graph. Progress is streamed.
Data is cached under the local project `data/` directory and can be reused while the VM survives.

Defaults: 3 batches of 8 queries, fanouts `[4, 2]`, 64 channels, 2 layers, and 10 timing repetitions.
Do not increase the workload before this small diagnostic passes. The report saves completed batches
incrementally; a failure also writes its error to JSON. The task's SQL/configured horizon is **91 days**,
despite an inconsistent two-year description in the hosted metadata.

In [ ]:
run_logged(
    [
        sys.executable,
        "-m",
        "trgnn.real_batch",
        "--download",
        "--data-dir",
        str(PROJECT_ROOT / "data/rel-stack"),
        "--device",
        DEVICE,
        "--batch-size",
        "8",
        "--batches",
        "3",
        "--channels",
        "64",
        "--fanouts",
        "4",
        "2",
        "--repeats",
        "10",
        "--output",
        str(RUN_DIR / "real-batch.json"),
    ],
    "real-batch.log",
)

Inspect the status. Success requires actual record-time visibility, disjoint edge ownership,
BFS prefix checks, matching logits/loss, and all three paired forward comparisons. Failure is
useful diagnostic information; it must not be described as a completed real-data result.

In [ ]:
report_path = RUN_DIR / "real-batch.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print("Status:", report["status"])
    print("Scope:", report["scope"])
    print("Trained:", report["trained"], "End-to-end:", report["end_to_end"])
    if "error" in report:
        print("Error:", report["error"])
    for result in report["batches"]:
        print("Batch", result["index"], "median ms:", result["median_ms"])
        print("Max logit errors:", result["max_absolute_logit_error"])
else:
    print("No real-batch report yet; export the available setup logs.")

## Download the evidence

Run this even if an earlier cell failed. The ZIP includes run reports and logs, not the full database.
Keep the archive before deleting/disconnecting the VM. Share the exported files for review.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/trgnn-real-results-" + run_id, "zip", root_dir=str(RUN_DIR))
files.download(archive)